# Scenarios and weights

Base, Upside and Downside paths are internally generated assumptions. IFRS 9 does not prescribe their names or probabilities.

In [1]:
from pathlib import Path
import json
import sqlite3
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score, brier_score_loss, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd().resolve()
if not (ROOT / "config").exists():
    ROOT = ROOT.parent
DB = ROOT / "database" / "ifrs9_ecl.sqlite3"
CFG = yaml.safe_load((ROOT / "config" / "project.yaml").read_text())

def query(sql):
    with sqlite3.connect(DB) as connection:
        return pd.read_sql_query(sql, connection)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
plt.rcParams["figure.figsize"] = (9, 4)

In [2]:
weights = query('select * from scenario_weight_analysis'); weights

,scenario,configured_weight,historical_analogue_frequency,configured_component,historical_component,selected_weight,weighting_method
0,Base,0.6000,0.4303,judgemental starting probability,nearest historical macro-regime frequency,0.5152,blended_historical_analogue
1,Downside,0.2000,0.1345,judgemental starting probability,nearest historical macro-regime frequency,0.1672,blended_historical_analogue
2,Upside,0.2000,0.4352,judgemental starting probability,nearest historical macro-regime frequency,0.3176,blended_historical_analogue


In [3]:
blend = CFG['scenario_weighting']['configured_weight_share']
weights[['scenario','configured_weight','historical_analogue_frequency','selected_weight']].assign(
recalculated_weight=lambda x: blend*x.configured_weight +
                            (1-blend)*x.historical_analogue_frequency,
difference=lambda x: x.recalculated_weight-x.selected_weight)

,scenario,configured_weight,historical_analogue_frequency,selected_weight,recalculated_weight,difference
0,Base,0.6000,0.4303,0.5152,0.5152,0.0000
1,Downside,0.2000,0.1345,0.1672,0.1672,0.0000
2,Upside,0.2000,0.4352,0.3176,0.3176,0.0000


In [4]:
query('''select scenario,
avg(case when month_number<=12 then unemployment_rate end) unemployment_first_year,
avg(case when month_number<=12 then gdp_growth_yoy end) gdp_first_year,
avg(case when month_number<=12 then hpi_growth_yoy end) hpi_first_year,
avg(case when month_number<=12 then mortgage_rate end) mortgage_rate_first_year
from macro_scenarios group by scenario''')

,scenario,unemployment_first_year,gdp_first_year,hpi_first_year,mortgage_rate_first_year
0,Base,4.3516,2.0462,2.5676,3.9900
1,Downside,6.8516,-0.9538,-5.4324,5.2400
2,Upside,3.6016,3.0462,4.5676,3.4900


Production weights blend the configured judgement and historical analogue frequency in equal proportions. The result is a modelling choice, not an official forecast or an IFRS 9 requirement. The mortgage-rate paths provide context; contractual loan rates remain the EIR approximation and EAD is scenario-invariant.

In [5]:
query('select * from scenario_summary order by scenario')

,scenario,scenario_weight,scenario_ecl,weighted_contribution
0,Base,0.5152,"337,567.2329","173,900.7725"
1,Downside,0.1672,"372,053.4097","62,221.1570"
2,Upside,0.3176,"329,484.2311","104,645.4807"
